# Fresh Kaggle T4 x2 Run: Fix 6

Use a fresh Kaggle notebook/session with **Internet ON** and **Accelerator: T4 x2**.

This notebook runs Fix 6: baseline head-to-head for HCPC-v2, CRAG, RAPTOR-2L, and optionally Self-RAG.

Recommended path: run the no-Self-RAG path first, package/download it, then attempt the Self-RAG smoke test and full Self-RAG run if the smoke test passes.

Expected output files after packaging:

- `/kaggle/working/fix6_t4x2_outputs.zip`
- `/kaggle/working/AAA_FIX6_T4X2_OUTPUTS.zip`


In [ ]:
import os
import pathlib
import subprocess
import time

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'rag-hallucination-detection'
REMOTE = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'

def run(cmd, cwd=None, check=True):
    print('\n>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)

print('START', time.ctime(), flush=True)
run(['bash', '-lc', 'pkill -f kaggle_stream_fix6_t4x2 || true; pkill -f kaggle_fix6_t4x2 || true; pkill -f fix_06_baseline_h2h_pareto.py || true; pkill -x ollama || true'], check=False)
run(['nvidia-smi'], check=False)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--progress', '--branch', 'main', REMOTE, str(REPO)], cwd=WORK)
else:
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=False)
    run(['git', '-C', str(REPO), 'checkout', 'main'], check=False)
    run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=False)

os.chdir(REPO)
run(['git', 'log', '-1', '--oneline'])
run(['bash', '-n', 'scripts/kaggle_fix6_t4x2.sh'])
run(['python', '-m', 'py_compile', 'scripts/kaggle_stream_fix6_t4x2.py', 'experiments/fix_06_baseline_h2h_pareto.py', 'src/selfrag_wrapper.py'])
print('Repo ready:', REPO, flush=True)


## 1. Setup + Ollama Preflight

This installs dependencies, installs/verifies Ollama, starts one Ollama server on GPU0 / port `11434`, pulls `mistral`, and checks preflight. GPU1 is left for Python-side models such as the detector and Self-RAG.


In [ ]:
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage setup --heartbeat 15

## 2. Recommended Run: Fix 6 Without Self-RAG

Run this first. It evaluates HCPC-v2, CRAG, and RAPTOR-2L on SQuAD and HotpotQA, then packages immediately so the output survives even if the runtime resets later.


In [ ]:
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage no_selfrag --heartbeat 30
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix6_t4x2_outputs.zip /kaggle/working/AAA_FIX6_T4X2_OUTPUTS.zip

## 3. Optional Self-RAG Smoke Test

Run this before the full Self-RAG path. It uses `n=5`, `max_contexts=50`, and `--selfrag_8bit`. If Hugging Face access, bitsandbytes, or VRAM fails, stop here and keep the no-Self-RAG package.


In [ ]:
!SELFRAG_QUANT=8bit python -u scripts/kaggle_stream_fix6_t4x2.py --stage smoke_selfrag --heartbeat 30

## 4. Optional Full Self-RAG Run

Only run this if the smoke test passes. It reruns Fix 6 with Self-RAG included and packages immediately after completion.


In [ ]:
!SELFRAG_QUANT=8bit python -u scripts/kaggle_stream_fix6_t4x2.py --stage selfrag --heartbeat 30
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix6_t4x2_outputs.zip /kaggle/working/AAA_FIX6_T4X2_OUTPUTS.zip

## 5. Status

Use this after a run or after an interruption. It prints row counts and output paths.


In [ ]:
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage status --heartbeat 15

## 6. Package Outputs

Run this manually if needed. The run cells above already package automatically after success.


In [ ]:
!python -u scripts/kaggle_stream_fix6_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix6_t4x2_outputs.zip /kaggle/working/AAA_FIX6_T4X2_OUTPUTS.zip
!find /kaggle/working -maxdepth 2 \( -name '*FIX6*.zip' -o -name '*fix6*.zip' \) -print

## 7. Direct Download Link

Run this after package succeeds if the Kaggle file browser is awkward.


In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import base64

paths = [
    Path('/kaggle/working/AAA_FIX6_T4X2_OUTPUTS.zip'),
    Path('/kaggle/working/fix6_t4x2_outputs.zip'),
]

p = next((x for x in paths if x.exists()), None)
if p is None:
    raise FileNotFoundError('No Fix 6 output zip found in /kaggle/working')
print('Using:', p, 'size:', round(p.stat().st_size / 1024 / 1024, 2), 'MB')

data = base64.b64encode(p.read_bytes()).decode()
display(HTML(f'''
<a download="fix6_t4x2_outputs.zip"
   href="data:application/zip;base64,{data}"
   style="font-size:22px;font-weight:700;color:#0b57d0">
   CLICK HERE TO DOWNLOAD fix6_t4x2_outputs.zip
</a>
'''))


## Debug If Anything Fails

Run this only if setup or a run stage fails.


In [ ]:
!echo '--- git ---'
!git -C /kaggle/working/rag-hallucination-detection log -1 --oneline || true
!echo '--- zips/csvs ---'
!find /kaggle/working /kaggle/input -maxdepth 20 -type f \( -iname '*FIX6*.zip' -o -iname '*fix6*.zip' -o -path '*/fix_06/*.csv' \) -print -exec ls -lh {} \; 2>/dev/null | head -n 240 || true
!echo '--- rows ---'
!python - <<'PY'
from pathlib import Path
import csv
root = Path('/kaggle/working/rag-hallucination-detection/data/revision/fix_06')
for p in sorted(root.glob('*.csv')):
    try:
        with p.open(newline='') as f:
            n = max(0, sum(1 for _ in csv.reader(f)) - 1)
    except Exception as exc:
        n = f'ERR:{exc}'
    print(p, n)
PY
!echo '--- processes ---'
!ps aux | grep -E 'ollama|fix6|selfrag|kaggle_stream' | grep -v grep || true
!echo '--- gpu ---'
!nvidia-smi || true
!echo '--- wrapper ---'
!tail -n 180 /kaggle/working/fix6_t4x2_wrapper.log 2>/dev/null || true
!echo '--- fix6 no selfrag ---'
!tail -n 180 /kaggle/working/rag-hallucination-detection/logs/revision/fix_06_baselines_no_selfrag.log 2>/dev/null || true
!echo '--- fix6 selfrag ---'
!tail -n 180 /kaggle/working/rag-hallucination-detection/logs/revision/fix_06_baselines_with_selfrag.log 2>/dev/null || true
!echo '--- ollama gpu0 ---'
!tail -n 140 /kaggle/working/fix6_t4x2_logs/ollama_gpu0.log 2>/dev/null || true
